# Brain extraction(Skull stripping)
How to do quick brain extraction using ants(antspynet module)

In [1]:
import os
from helpers import *

import ants
import SimpleITK as sitk

print(f'AntsPy version = {ants.__version__}')
print(f'SimpleITK version = {sitk.__version__}')

AntsPy version = 0.6.3
SimpleITK version = 2.4.0


In [2]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
print(f'project folder = {BASE_DIR}')

project folder = /home/folkk2/img_group_project/test-MRI-preprocessing-technique


In [3]:
raw_examples = [
    'fsl-open-dev_sub-001_T1w.nii.gz',
    'wash-120_sub-001_T1w.nii.gz',
    'kf-panda_sub-01_ses-3T_T1w.nii.gz',
    'listen-task_sub-UTS01_ses-1_T1w.nii.gz',
    'brain-lesion_T1w.nii.gz'
]

In [5]:
raw_example = raw_examples[4]
raw_img_path = os.path.join(BASE_DIR, 'assets', 'raw_examples', raw_example)
raw_img_ants = ants.image_read(raw_img_path, reorient='IAL')


explore_3D_array(arr=raw_img_ants.numpy())

interactive(children=(IntSlider(value=95, description='SLICE', max=191), Output()), _dom_classes=('widget-inte…

### Deep Learning based method
### Load Model via AntsPyNet API and predict

In [6]:
from antspynet.utilities import brain_extraction

I0000 00:00:1773150303.649975   67606 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [7]:
prob_brain_mask = brain_extraction(raw_img_ants,modality="t1",verbose=True)

Brain extraction:  retrieving model weights.
5683832/5683832 ━━━━━━━━━━━━━━━━━━━━ 52s 9us/step
Brain extraction:  retrieving template.
14969865/14969865 ━━━━━━━━━━━━━━━━━━━━ 8s 1us/step


W0000 00:00:1773150383.472501   68314 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1773150383.758094   67606 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Brain extraction:  normalizing image to the template.
Brain extraction:  prediction and decoding.
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Brain extraction:  renormalize probability mask to native space.


### Inspect probabilities array

In [8]:
print(prob_brain_mask)
explore_3D_array(prob_brain_mask.numpy())

ANTsImage (IAL)
	 Pixel Type : float (float32)
	 Components : 1
	 Dimensions : (192, 192, 160)
	 Spacing    : (1.25, 1.25, 1.2)
	 Origin     : (98.1114, -149.1525, -129.5975)
	 Direction  : [-0.  0. -1.  0.  1.  0.  1.  0.  0.]



interactive(children=(IntSlider(value=95, description='SLICE', max=191), Output()), _dom_classes=('widget-inte…

### Generate final mask

In [9]:
brain_mask = ants.get_mask(prob_brain_mask, low_thresh=0.5)

In [10]:
explore_3D_array_with_mask_contour(raw_img_ants.numpy(), brain_mask.numpy())

interactive(children=(IntSlider(value=95, description='SLICE', max=191), Output()), _dom_classes=('widget-inte…

In [11]:
out_folder =  os.path.join(BASE_DIR, 'assets', 'preprocessed')
out_folder = os.path.join(out_folder, raw_example.split('.')[0]) # create folder with name of the raw file
os.makedirs(out_folder, exist_ok=True) # create folder if not exists

out_filename = add_suffix_to_filename(raw_example, suffix='brainMaskByDL')
out_path = os.path.join(out_folder, out_filename)

print(raw_img_path[len(BASE_DIR):])
print(out_path[len(BASE_DIR):])

/assets/raw_examples/brain-lesion_T1w.nii.gz
/assets/preprocessed/brain-lesion_T1w/brain-lesion_T1w_brainMaskByDL.nii.gz


In [12]:
brain_mask.to_file(out_path)

### Generate brain masked

In [13]:
masked = ants.mask_image(raw_img_ants, brain_mask)

explore_3D_array(masked.numpy())

interactive(children=(IntSlider(value=95, description='SLICE', max=191), Output()), _dom_classes=('widget-inte…

In [14]:
out_filename = add_suffix_to_filename(raw_example, suffix='brainMaskedByDL')
out_path = os.path.join(out_folder, out_filename)

print(raw_img_path[len(BASE_DIR):])
print(out_path[len(BASE_DIR):])

/assets/raw_examples/brain-lesion_T1w.nii.gz
/assets/preprocessed/brain-lesion_T1w/brain-lesion_T1w_brainMaskedByDL.nii.gz


In [15]:
masked.to_file(out_path)